# Homework 4 Notebook
---

## Overview

In this notebook, you will **fine-tune a pre-trained BERT-style model** on a binary safety classification task. Given an input text, your model should predict whether it is **safe (label 0)** or **unsafe (label 1)**.

This notebook will:
- Load your training data
- Tokenize and batching inputs
- Fine-tune `distilbert-base-uncased` on your dataset
- Evaluate on the provided validation set each epoch
- Log all metrics to **Weights & Biases (wandb)**
- Save model checkpoints to an `artifacts/` folder

**Your job**: Create a high-quality training dataset and run this notebook to train the best model you can.

**Note**: You should not modify any part of the code in this notebook, as the focus of this homework is on better training dataset curation. The only part of the code you can modify is in **Step 1**, where you can change run_name, wandb configs, and training hyperparameters, and **Step 10**, where you can change the inputs to the model to check its behavior.

---

## Hardware

Make sure you're using a **T4 GPU** (or better) in Colab:
- Go to **Runtime → Change runtime type → T4 GPU**

**Estimated training times** (5 epochs, 500 training examples, sequence length 128):

| Hardware | Time per Epoch | 5 Epochs Total |
|---|---|---|
| CPU only | ~3–4 min | ~15–20 min |
| T4 GPU (Colab free tier) | ~5–6 sec | **~1 min** |

> Always use the T4 GPU. CPU training is much slower for this model.

---

## Data Format

Your training and validation data must be a **`.jsonl` file** (JSON Lines): one JSON object per line, with **two required keys**:

```json
{"text": "Hi Bob, meet for lunch?", "label": 0}
{"text": "Hi Bob, meet for early breakfast?", "label": 1}
{"text": "Hi Bob, thinking of getting into exercise, want to come to the gym with me?", "label": 1}
{"text": "Hi Bob, Should we set up a meeting to discuss next steps?", "label": 0}
```

| Key | Type | Description |
|---|---|---|
| `text` | `str` | The text to classify |
| `label` | `int` | `0` = safe, `1` = unsafe |

> **Common mistakes**: Make sure every line is valid JSON. No trailing commas. No outer `[...]` brackets. Each line is its own object.

---

## Weights & Biases (wandb) Setup

We use wandb to track your experiments. Follow these steps:

1. Go to [https://wandb.ai](https://wandb.ai) and create a free account.
2. Once logged in, click your profile icon → **User Settings** → copy your **API key**.
3. Paste the API key to WANDB_API_KEY in **Step 1**.
4. Configure your RUN_NAME and WANDB_PROJECT name in **Step 1**.
5. Sign in to wandb in **Step 3**.
5. Start training your models.
6. Your runs will appear at `https://wandb.ai/<your-username>/<project-name>`.

> Each training run creates a new wandb "run". Give your runs descriptive names (see `RUN_NAME` below) to compare strategies.

---

## Step 0: Install Dependencies and set up


In [1]:
# Install required packages
# This cell only needs to run once per Colab session
!pip install -q transformers wandb scikit-learn IPython python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.9 MB/s eta 0:00:0000:01


In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
%cd "/content/drive/MyDrive/hw4-release"

import sys
from importlib import reload
# Create a fake 'imp' module with just the reload function
class ImpModule:
    reload = staticmethod(reload)

sys.modules['imp'] = ImpModule()


import IPython

ipython = IPython.get_ipython()
ipython.run_line_magic("sx", f"chmod +x scripts/*.py")

%load_ext autoreload
%autoreload 2

## Step 1: Configuration

**Edit the variables in this cell** before running anything else.

- `TRAIN_DATA_PATH`: path to your training `.jsonl` file
- `VAL_DATA_PATH`: path to the course-provided validation `.jsonl` file
- `RUN_NAME`: a short description of this experiment (e.g. `"synthetic-gpt4-v1"`, `"augmented-v2"`)
- `WANDB_PROJECT`: the wandb project name — keep this consistent across all your runs so they appear together

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

# ============================================================
#   EDIT THESE SETTINGS BEFORE RUNNING
# ============================================================

TRAIN_DATA_PATH = "data/email_dataset_train.jsonl"      # path to your training data
VAL_DATA_PATH   = "data/email_dataset_dev.jsonl"         # path to your validation data

RUN_NAME        = "gpt4o-mini-v1"           # short description of this run
WANDB_API_KEY   = os.getenv("WANDB_API_KEY")            # loaded from .env file
WANDB_PROJECT   = "safety-classifier"          # wandb project name (keep consistent!)



MODEL_NAME      = "distilbert/distilbert-base-uncased"    # do not change
MAX_SEQ_LEN     = 128                          # max tokens per input; increase if inputs are long

# ============================================================
#  Hyperparameters — you may tune these
# ============================================================

BATCH_SIZE      = 32                           # reduce to 16 if you get CUDA OOM errors
NUM_EPOCHS      = 5                            # number of training epochs
LEARNING_RATE   = 2e-5                         # standard lr for fine-tuning BERT
WEIGHT_DECAY    = 0.01                         # L2 regularization
WARMUP_RATIO    = 0.1                          # fraction of steps used for lr warmup
SEED            = 42

DEVICE         = "cuda" #change to 'cpu' if no GPUs available

# ============================================================
#  Output directory for saved checkpoints
# ============================================================

ARTIFACTS_DIR   = "artifacts"        # checkpoints saved here

print(" Configuration loaded.")
print(f"   Model       : {MODEL_NAME}")
print(f"   Run name    : {RUN_NAME}")
print(f"   Epochs      : {NUM_EPOCHS}")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   LR          : {LEARNING_RATE}")
print(f"   Output dir  : {ARTIFACTS_DIR}")

 Configuration loaded.
   Model       : distilbert/distilbert-base-uncased
   Run name    : gpt4o-mini-v1
   Epochs      : 5
   Batch size  : 32
   LR          : 2e-05
   Output dir  : artifacts


## Step 2: Imports & Reproducibility

In [6]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report

import wandb

# ── Reproducibility ──────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device(DEVICE)
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   No GPU found — training will be very slow. Switch to T4 GPU runtime.")

# ── Create artifacts directory ────────────────────────────────
Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)
print(f"   Artifacts dir: {ARTIFACTS_DIR}")

Using device: cuda
   GPU: NVIDIA L4
   VRAM: 23.7 GB
   Artifacts dir: artifacts


## Step 3: Login to Weights & Biases

Remember to set your WANDB_API_KEY in **Step 1**.

In [7]:
wandb.login(key=WANDB_API_KEY)
print("Logged in to wandb.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhysq (dhysq-cornell-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in to wandb.


## Step 4: Load Data

This cell reads your `.jsonl` files and does basic sanity checks.

If you see an error here, check:
- The file paths in Step 1
- That each line has both `"text"` and `"label"` keys
- That labels are `0` or `1` (integers). `0` corresponds to safe text. `1` corresponds to unsafe text.

In [8]:
def load_jsonl(path):
    """Load a .jsonl file into a list of dicts."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue  # skip blank lines
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON parse error on line {i} of {path}: {e}")
            assert "text" in record, f"Missing 'text' key on line {i} of {path}"
            assert "label" in record, f"Missing 'label' key on line {i} of {path}"
            assert record["label"] in (0, 1), (
                f"Label must be 0 or 1, got {record['label']} on line {i} of {path}"
            )
            records.append(record)
    return records


# Load splits
train_data = load_jsonl(TRAIN_DATA_PATH)
val_data   = load_jsonl(VAL_DATA_PATH)

# Label distributions
def label_dist(data, split):
    labels = [d["label"] for d in data]
    n0 = labels.count(0)
    n1 = labels.count(1)
    print(f"  {split}: {len(data)} examples  |  safe (0): {n0} ({100*n0/len(data):.1f}%)  |  unsafe (1): {n1} ({100*n1/len(data):.1f}%)")

print("Dataset statistics:")
label_dist(train_data, "Train")
label_dist(val_data,   "Val  ")

print()
print("First 3 training examples:")
for ex in train_data[:3]:
    print(f"   [{ex['label']}] {ex['text'][:100]}..." if len(ex['text']) > 100 else f"   [{ex['label']}] {ex['text']}")

Dataset statistics:
  Train: 1650 examples  |  safe (0): 842 (51.0%)  |  unsafe (1): 808 (49.0%)
  Val  : 200 examples  |  safe (0): 98 (49.0%)  |  unsafe (1): 102 (51.0%)

First 3 training examples:
   [0] Bob, Did you finish the report I sent? Thanks
   [1] Bob, Can we meet for breakfast at 7 AM? Thanks
   [1] Can we meet for a quick chat at 9 AM?


## Step 5: Tokenization & Dataset

We use HuggingFace's `AutoTokenizer` for `distilbert-base-uncased`. It converts raw text into token IDs, attention masks, and (optionally) token type IDs that the model expects.

In [9]:
# ── Load tokenizer ────────────────────────────────────────────
print(f"Loading tokenizer for '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size:,}")


# ── PyTorch Dataset ───────────────────────────────────────────
class SafetyDataset(Dataset):
    """
    Wraps a list of {input, label} dicts into a PyTorch Dataset.
    Tokenizes text on-the-fly for simplicity.
    """
    def __init__(self, records, tokenizer, max_len):
        self.records   = records
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(
            rec["text"],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(rec["label"], dtype=torch.long),
        }


train_dataset = SafetyDataset(train_data, tokenizer, MAX_SEQ_LEN)
val_dataset   = SafetyDataset(val_data,   tokenizer, MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

Loading tokenizer for 'distilbert/distilbert-base-uncased'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded. Vocab size: 30,522
Train batches: 52  |  Val batches: 7


## Step 6: Load Model

We load `distilbert-base-uncased` with a **binary classification head** (2 output labels) on top.

In [10]:
print(f"Loading model '{MODEL_NAME}'...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "safe", 1: "unsafe"},
    label2id={"safe": 0, "unsafe": 1},
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded.")
print(f"   Total parameters    : {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

Loading model 'distilbert/distilbert-base-uncased'...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
   Total parameters    : 66,955,010
   Trainable parameters: 66,955,010


## Step 7: Optimizer & Scheduler

We use **AdamW** (the standard optimizer for fine-tuning BERT-style models) with a **linear warmup + linear decay** learning rate schedule.

In [11]:
total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Optimizer ready.")
print(f"   Total training steps : {total_steps}")
print(f"   Warmup steps         : {warmup_steps}")

Optimizer ready.
   Total training steps : 260
   Warmup steps         : 26


## Step 8: Training

In this step, we will run the main training loop. For each epoch:
1. Train on all batches → log **train loss** and **train accuracy**
2. Evaluate on the validation set → log **val accuracy** and **val F1**
3. Save a checkpoint to `artifacts/`
4. Log everything to wandb

The **best checkpoint** (by val accuracy) is saved as `artifacts/best_model/`.

> Watch your wandb dashboard at [https://wandb.ai](https://wandb.ai) while this runs.

In [12]:
# ── Helper: evaluate on a DataLoader ─────────────────────────
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="binary", zero_division=0)

    per_class_stat = classification_report(
      all_labels, all_preds,
      target_names=["class (0)", "class (1)"],
      digits=4,
      output_dict=True,
    )

    return avg_loss, acc, f1, all_preds, all_labels, per_class_stat

In [13]:
# ── Initialize wandb run ──────────────────────────────────────
run = wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        "model":         MODEL_NAME,
        "max_seq_len":   MAX_SEQ_LEN,
        "batch_size":    BATCH_SIZE,
        "num_epochs":    NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay":  WEIGHT_DECAY,
        "warmup_ratio":  WARMUP_RATIO,
        "train_size":    len(train_data),
        "val_size":      len(val_data),
        "seed":          SEED,
    },
)
print(f"wandb run initialized: {run.url}")


# ── Training loop ─────────────────────────────────────────────
best_val_acc   = 0.0
best_epoch     = -1
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    # ── Train ──────────────────────────────────────────────────
    model.train()
    train_loss_total = 0.0
    train_preds_all, train_labels_all = [], []

    for step, batch in enumerate(train_loader, start=1):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_loss_total += loss.item()
        preds = outputs.logits.argmax(dim=-1).detach().cpu().numpy()
        train_preds_all.extend(preds)
        train_labels_all.extend(labels.detach().cpu().numpy())

        # Log step-level loss to wandb
        global_step = (epoch - 1) * len(train_loader) + step
        wandb.log({"train/step_loss": loss.item(), "train/step": global_step})

        if step % 5 == 0 or step == len(train_loader):
            print(f"  Step {step:>3}/{len(train_loader)} | loss: {loss.item():.4f} | lr: {scheduler.get_last_lr()[0]:.2e}")

    # Epoch-level train metrics
    avg_train_loss = train_loss_total / len(train_loader)
    train_acc      = accuracy_score(train_labels_all, train_preds_all)
    train_f1       = f1_score(train_labels_all, train_preds_all, average="binary", zero_division=0)

    classes = ["class (0)", "class (1)"]
    train_per_class_stat = classification_report(
      train_labels_all, train_preds_all,
      target_names=classes,
      digits=4,
      output_dict=True,
    )

    # ── Validate ───────────────────────────────────────────────
    val_loss, val_acc, val_f1, val_preds, val_labels, val_per_class_stat = evaluate(model, val_loader, DEVICE)

    epoch_time = time.time() - epoch_start

    print(f"\n  Epoch {epoch} results:")
    print(f"     Train loss : {avg_train_loss:.4f}  |  Train acc: {train_acc:.4f}  |  Train F1: {train_f1:.4f}")
    print(f"     Val   loss : {val_loss:.4f}  |  Val   acc: {val_acc:.4f}  |  Val   F1: {val_f1:.4f}")

    wandb_keys = {
        "epoch":           epoch,
        "train/loss":      avg_train_loss,
        "train/accuracy":  train_acc,
        "train/f1":        train_f1,
        "val/loss":        val_loss,
        "val/accuracy":    val_acc,
        "val/f1":          val_f1,
        "epoch_time_sec":  epoch_time,
        "learning_rate":   scheduler.get_last_lr()[0],
    }

    for c in classes:
      print(f'     Train {c} : Prec: {train_per_class_stat[c]["precision"]:.4f}  |  Rec: {train_per_class_stat[c]["recall"]:.4f}  |  F1: {train_per_class_stat[c]["f1-score"]:.4f}')
      print(f'     Val {c} : Prec: {val_per_class_stat[c]["precision"]:.4f}  |  Rec: {val_per_class_stat[c]["recall"]:.4f}  |  F1: {val_per_class_stat[c]["f1-score"]:.4f}')
      wandb_keys[f"train/{c}_prec"] = train_per_class_stat[c]["precision"]
      wandb_keys[f"train/{c}_rec"] = train_per_class_stat[c]["recall"]
      wandb_keys[f"train/{c}_f1"] = train_per_class_stat[c]["f1-score"]
      wandb_keys[f"val/{c}_prec"] = val_per_class_stat[c]["precision"]
      wandb_keys[f"val/{c}_rec"] = val_per_class_stat[c]["recall"]
      wandb_keys[f"val/{c}_f1"] = val_per_class_stat[c]["f1-score"]

    print(f"     Epoch time : {epoch_time:.1f}s")



    # Log epoch-level metrics to wandb
    wandb.log(wandb_keys)

    # ── Save checkpoint ────────────────────────────────────────
    ckpt_dir = os.path.join(ARTIFACTS_DIR, f"epoch_{epoch}")
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    print(f"     Checkpoint saved → {ckpt_dir}")

    # ── Track best model ───────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        best_dir     = os.path.join(ARTIFACTS_DIR, "best_model")
        model.save_pretrained(best_dir)
        tokenizer.save_pretrained(best_dir)
        print(f"     New best model. Val acc: {best_val_acc:.4f} → saved to {best_dir}")


# ── Final summary ─────────────────────────────────────────────
total_time = time.time() - training_start
print(f"\n{'='*60}")
print(f"Training complete.")
print(f"   Total time  : {total_time/60:.1f} minutes")
print(f"   Best epoch  : {best_epoch}")
print(f"   Best val acc: {best_val_acc:.4f}")
print(f"   Best model  : {ARTIFACTS_DIR}/best_model")

wandb.summary["best_val_accuracy"] = best_val_acc
wandb.summary["best_epoch"]        = best_epoch
wandb.finish()
print(f"   wandb run finished.")

wandb run initialized: https://wandb.ai/dhysq-cornell-university/safety-classifier/runs/0s6qrmbl

Epoch 1/5
  Step   5/52 | loss: 0.6910 | lr: 3.85e-06
  Step  10/52 | loss: 0.6707 | lr: 7.69e-06
  Step  15/52 | loss: 0.7050 | lr: 1.15e-05
  Step  20/52 | loss: 0.6841 | lr: 1.54e-05
  Step  25/52 | loss: 0.6780 | lr: 1.92e-05
  Step  30/52 | loss: 0.6458 | lr: 1.97e-05
  Step  35/52 | loss: 0.6827 | lr: 1.92e-05
  Step  40/52 | loss: 0.6364 | lr: 1.88e-05
  Step  45/52 | loss: 0.5617 | lr: 1.84e-05
  Step  50/52 | loss: 0.4931 | lr: 1.79e-05
  Step  52/52 | loss: 0.6177 | lr: 1.78e-05

  Epoch 1 results:
     Train loss : 0.6476  |  Train acc: 0.6176  |  Train F1: 0.6246
     Val   loss : 0.4959  |  Val   acc: 0.7700  |  Val   F1: 0.8051
     Train class (0) : Prec: 0.6358  |  Rec: 0.5867  |  F1: 0.6103
     Val class (0) : Prec: 0.8939  |  Rec: 0.6020  |  F1: 0.7195
     Train class (1) : Prec: 0.6014  |  Rec: 0.6498  |  F1: 0.6246
     Val class (1) : Prec: 0.7090  |  Rec: 0.9314  | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

     Checkpoint saved → artifacts/epoch_1


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

     New best model. Val acc: 0.7700 → saved to artifacts/best_model

Epoch 2/5
  Step   5/52 | loss: 0.5618 | lr: 1.74e-05
  Step  10/52 | loss: 0.4187 | lr: 1.69e-05
  Step  15/52 | loss: 0.3275 | lr: 1.65e-05
  Step  20/52 | loss: 0.2835 | lr: 1.61e-05
  Step  25/52 | loss: 0.2933 | lr: 1.56e-05
  Step  30/52 | loss: 0.3039 | lr: 1.52e-05
  Step  35/52 | loss: 0.3103 | lr: 1.48e-05
  Step  40/52 | loss: 0.1101 | lr: 1.44e-05
  Step  45/52 | loss: 0.1364 | lr: 1.39e-05
  Step  50/52 | loss: 0.0683 | lr: 1.35e-05
  Step  52/52 | loss: 0.1678 | lr: 1.33e-05

  Epoch 2 results:
     Train loss : 0.2918  |  Train acc: 0.8958  |  Train F1: 0.8945
     Val   loss : 0.1723  |  Val   acc: 0.9500  |  Val   F1: 0.9500
     Train class (0) : Prec: 0.9046  |  Rec: 0.8895  |  F1: 0.8970
     Val class (0) : Prec: 0.9314  |  Rec: 0.9694  |  F1: 0.9500
     Train class (1) : Prec: 0.8869  |  Rec: 0.9022  |  F1: 0.8945
     Val class (1) : Prec: 0.9694  |  Rec: 0.9314  |  F1: 0.9500
     Epoch time 

...

     Checkpoint saved → artifacts/epoch_5


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

     New best model. Val acc: 0.9750 → saved to artifacts/best_model

Training complete.
   Total time  : 2.5 minutes
   Best epoch  : 5
   Best val acc: 0.9750
   Best model  : artifacts/best_model


epoch,▁▃▅▆█
epoch_time_sec,█▁▁▂▂
learning_rate,█▆▅▃▁
train/accuracy,▁▆███
train/class (0)_f1,▁▆███
train/class (0)_prec,▁▆███
train/class (0)_rec,▁▆███
train/class (1)_f1,▁▆███
train/class (1)_prec,▁▆███
train/class (1)_rec,▁▆███
+13,...


   wandb run finished.


## Step 9: Final Evaluation on Validation Set

Load the best saved model and print the full classification report.

In [14]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

best_dir = os.path.join(ARTIFACTS_DIR, "best_model")
print(f"Loading best model from: {best_dir}")

best_model     = AutoModelForSequenceClassification.from_pretrained(best_dir).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(best_dir)

# Re-create val loader with best tokenizer (should be identical, but let's be explicit)
final_val_dataset = SafetyDataset(val_data, best_tokenizer, MAX_SEQ_LEN)
final_val_loader  = DataLoader(final_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

_, final_acc, final_f1, final_preds, final_labels, _ = evaluate(best_model, final_val_loader, DEVICE)

print(f"\nFinal Validation Results (best model from {best_dir}):")
print(f"   Accuracy : {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"   F1 Score : {final_f1:.4f}")
print()
print("Classification Report:")
print(classification_report(
    final_labels, final_preds,
    target_names=["safe (0)", "unsafe (1)"],
    digits=4
))

Loading best model from: artifacts/best_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Final Validation Results (best model from artifacts/best_model):
   Accuracy : 0.9750 (97.50%)
   F1 Score : 0.9758

Classification Report:
              precision    recall  f1-score   support

    safe (0)     0.9895    0.9592    0.9741        98
  unsafe (1)     0.9619    0.9902    0.9758       102

    accuracy                         0.9750       200
   macro avg     0.9757    0.9747    0.9750       200
weighted avg     0.9754    0.9750    0.9750       200



## Step 10: Run Inference on Custom Examples

Use this cell to quickly test your model on new inputs.

In [15]:
def predict(texts, model, tokenizer, device, max_len=MAX_SEQ_LEN):
    """Run inference on a list of strings. Returns predicted labels and confidence scores."""
    model.eval()
    enc = tokenizer(
        texts,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    labels = probs.argmax(axis=-1)
    return labels, probs


# ── Edit these examples ────────────────────────────────────
test_inputs = [
    "Hi Bob, How do I make a birthday cake?",
    "run tmrw?",
    "Hi Bob,  Are you available for a quick call at 7 AM?  Thanks, Laura",
]

pred_labels, pred_probs = predict(test_inputs, best_model, best_tokenizer, DEVICE)

print("Inference Results:")
print(f"{'Text':<55} {'Pred':>6} {'P(safe)':>9} {'P(unsafe)':>11}")
print("-" * 85)
for text, label, probs in zip(test_inputs, pred_labels, pred_probs):
    pred_str = "unsafe" if label == 1 else "safe"
    short    = text[:52] + "..." if len(text) > 52 else text
    print(f"{short:<55} {pred_str:>6} {probs[0]:>9.3f} {probs[1]:>11.3f}")

Inference Results:
Text                                                      Pred   P(safe)   P(unsafe)
-------------------------------------------------------------------------------------
Hi Bob, How do I make a birthday cake?                    safe     0.981       0.019
run tmrw?                                               unsafe     0.008       0.992
Hi Bob,  Are you available for a quick call at 7 AM?... unsafe     0.009       0.991


## Step 11: Download Your Best Model

Run this cell to zip your best model checkpoint and download it from Colab.

> **For milestone submission**: download the zip file from this step and submit it via the course submission portal.

In [16]:
import shutil
from google.colab import files

train_file_path = "data/email_dataset_train.jsonl" #set as the path to the jsonl file contianing YOUR training data, not the course provided one.

# Zip the best model directory
zip_name = f"best_model_{RUN_NAME}"
zip_path = f"/artifacts/{zip_name}.zip"

dest_json_path = os.path.join(ARTIFACTS_DIR, "best_model", "train.jsonl")

# Copy the JSON file into best_model/
shutil.copy(train_file_path, dest_json_path)


shutil.make_archive(
    f"/artifacts/{zip_name}",
    "zip",
    ARTIFACTS_DIR,
    "best_model"
)

print(f"Zipped model to: {zip_path}")
print("   Downloading...")
files.download(zip_path)

Zipped model to: /artifacts/best_model_gpt4o-mini-v1.zip
   Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>